# jev-decision: stage 1 (Colab)
Runtime -> Change runtime type -> **GPU** (T4 works with LoRA, 8k context needs A100/L4). Builds public data, trains the head (+LoRA) on BGE-M3 and pushes the checkpoint to GitHub.

In [ ]:
GITHUB_REPO = "YOUR_USER/jev-decision"   # <- edit
BACKBONE = "BAAI/bge-m3"
!git lfs install
!git clone https://github.com/$GITHUB_REPO repo && cd repo && pip install -q -e .[data]
%cd repo

## 1. Build public stage-1 data
Sources that fail to load (gated/renamed) are skipped. For xlam tool-use data, accept the dataset terms on HF and `huggingface-cli login` first.

In [ ]:
!python scripts/build_public_data.py --out data --per_source 8000

In [ ]:
!python scripts/validate_jsonl.py data/stage1_train.jsonl --tokenizer $BACKBONE --max_length 1024

## 2. Train
LoRA keeps the checkpoint small enough for GitHub. Lower `--max_length`/batch if you hit OOM.

In [ ]:
!python -m jev.train --backbone $BACKBONE --train data/stage1_train.jsonl --val data/stage1_val.jsonl \
    --out checkpoints/stage1 --max_length 1024 --head_length 512 --lora 16 --bf16 --grad_ckpt \
    --batch_size 8 --grad_accum 4 --epochs 2 --lr_head 3e-4 --lr_encoder 1e-4

## 3. Check, then push
Look at `by_task` accuracy before pushing; compare with the majority-class baseline for each task.

In [ ]:
!cat checkpoints/stage1/metrics.json && du -sh checkpoints/stage1/*

In [ ]:
import getpass, os
os.environ["GH_TOKEN"] = getpass.getpass("GitHub token (repo scope): ")
!git config user.email "you@example.com" && git config user.name "your-name"
!git add checkpoints data/DATA_LICENSES.md data/stage1_val.jsonl && git commit -m "stage-1 checkpoint" 
!git push https://$GH_TOKEN@github.com/$GITHUB_REPO HEAD

Training data (`stage1_train.jsonl`) is deliberately not committed: it is regenerable and its licences differ. Commit it too only after checking each licence in `data/DATA_LICENSES.md`.